# RAG 평가

RAG를 만들었다고 했을 때 이게 실제로 잘 작동하는지 어떻게 알아내야 할까요? 이를 측정하는 방법은 두가지가 있습니다.

1. 검색(retrieval) 성능평가 : 관련 컨텍스트를 얼마나 잘 찾아오는가
2. QA 자체의 성능 평가 : 결과적으로 질문에 좋은 답을 주는가 (이게 사실 비즈니스 목표에 가장 직결됨)

청킹(chunking) 전략과 인코더 등을 실험하면서 이것들이 평가 결과에 어떤 영향을 주는지 알아보겠습니다.

RAG의 장점 :
1. 매우 빠르게 구축 가능 : 뭔가에 대해서 다 아는 QA 시스템을 몇 분 안에 만들 수 있음
2. 확장성이 좋음 : 데이터를 100배 늘려도 시간은 좀 더 걸리지만 하룻밤 안에 처리 가능. 백터 스토아가 있으면 쿼리 자체는 매우 빠름
3. 파인튜닝과 대조되는 지점 : 파인튜닝은 힘들고, 오래 걸리고, 비용이 많이 드는 반면 RAG는 빠름
4. 컨텍스트 길이를 절약 : 관련 있는 부분만 골라서 넣기 때문에 불필요한 정보로 컨텍스트를 오염시키지 않고, LLM이 정확한 답을 낼 수 있도록 컨텍스트 창을 가장 관련성 높은 정보에 집중 시킬 수 있음

## RAG의 한계
RAG는 본질적으로 트랜스포머에 붙인 '해킹'입니다.

트랜스포머 신경망은 입력에서 뭐가 중요한지 스스로 주의(attention)를 기울이도록 학습되어 있고, 실제로 이걸 잘합니다. 문제는 우리가 정보량이 너무 많아서, 트랜스포마가 굳이 별로 중요하지 않은 부분까지 다 처리하며 "바퀴를 헛도는"걸 원치 않는다는 것입니다.

그래서 인코더와 백터 유사도를 이용해 "이 부분이 관련 있어 보인다"를 미리 걸래내는 일종의 '지름길(해킹)'로 RAG를 쓰는 겁니다. 벡터 공간에서 멀리 떨어진 청크는 관련 없다고 간주하고 버리는데, 사실은 관련이 있는 걸 놓칠 수도 있습니다. 이건 결국 성능 최적화를 위한 트릭이고, 근본적으로 과학적인 방법이 아닌 해킹이라는 걸 받아 드려야 합니다.

### 해킹이라서 생기는 문제 :

* 항상 잘 작동하지 않음
* 매우 실험적(경험적) : "이런 입력엔 어떤 인코더/청킹/RAG 기법을 써야 하나요?"라는 질문에 만든사람도 추측만 할 수 있고, 직접 해봐야 알 수 있음
* 다 제대로 했다고 생각해도 예상치 못하게 이상한 답이 나올 수 있음 (예: 어떤 상을 누가 받았는지 묻는 질문에 오답) -> 파고들어 보면 청킹 전략 때문에 잘못된 정보가 검색된 것으로 드러남.
* 한 문제를 고치면 다른 데서 또 문제가 튀어나오는 "두더지 잡기(whack-a-mole)" 같은 반복 - 다소 지저분한 과정
* 좋은 아이디어라고 생각해서 열심히 만들었는데 실제 질문엔 엉뚱한 답이나 "모른다"는 답이 나오는 경우도 흔함

### 청킹(Chunking) 예시로 보는 실험적 특성
문서 전체를 백터 스토어에 놓지 않고 조각(청크)으로 나눠서 각각 벡터화 하는 이유 : 질문은 보통 문서 전체가 아니라 일부에만 해당되기 때문입니다.

* 청크가 너무 크면 : 한 청크에 너무 많은 내용이 섞여서 구분력이 떨어짐
* 청크가 너무 작으면 : ("누가 IOT상을 받았나"라는 질문은 그녀의 전체 이력과는 매칭이 잘 안 될 수 있어서 청크가 잘게 나눠야 함) - 근데 너무 잘게 나누면 정작 그 청크 안에 이름 자체가 빠져 있을 수 있음. 청크가 선택돼도 답을 낼 수 있는 정보가 없는 상황이 생김

그래서 청크 크기를 키워야 할 때도, 줄여야 할때도, 청크 간 겹침(overlap)을 늘려야 할 때도 있고 이런 조정들이 각각 RAG성능에 영향을 줍니다. 마치 연금술 같지만, 다행히 이걸 과학적으로 다루는 방법이 있는데 바로 평가(evaluation) 체계를 만드는 것 입니다.

## 결론 : 왜 평가가 중요한가
LLM 작업 전반에 반복되는 주제와 같습니다. - 매우 실험적이고 반복적이며 프롬프트를 이것저것 시도해보는 과정이라 답답하게 느껴질 수 있습니다. 이를 해소하는 방법은 설정(setup)의 성능을 정량적으로 측정할 평가 지표를 만드는 것입니다. 그러면 실험하고 반복하면서 개선되고 있는지 나빠지고 있는지를 측정할 수 있는 북극성 지표가 생기고, 그 지표가 오를 때까지 반복 개선하면 됩니다. 이게 바로 연금술 같은 RAG 튜닝 과정에 절차를 부여하는 방법 입니다.


# 평가(evaluation) 구축의 3단계

## 1단계 : 골든 데이터셋 만들기

성과를 측정할 기준이 될 질문-답변 세트를 큐레이션(의도를 가지고 선별하고, 다듬고, 구성하는 것)하는 것이 첫 단계입니다.
방법은 여러가지지만, 일반적인 방식은 :
* 질문 목록을 만들고
* 각 질문에 대해 정답 컨텍스트에 반드시 포함되어야 할 키워드를 몇 개 지정
* 이걸로 검색된 청크에 그 키워드가 들어있는지 테스트

예시 : "권위 있는 IOT 상을 누가 받았나?" 라는 질문이라면 -> 수상자 이름(예: "Bruno Song"의 이름과 성)이 키워드가 되어야 함. 참고용 완벽한 정답 문장("Bruno Song이 권위 있는 IoT 상을 수상했다") 도 함께 준비

질문은 어디서 구하나?

1. 직접 데이터를 보고 생각해내기 - "이 질문에 시스템이 답할 수 있으면 만족스럽다" 싶은 것들
2. 더 좋은 방법 : 실제 사용자 피드백/기존 질문 이력 - 실제 운영 중인 시스템을 개선하는 상황이라면, 예전에 이메일로 들어와서 사람이 24시간 걸려 답했던 질문 + 전문가 답변이 있을것. 이게 골든 데이터의 최고의 원천이며 양도 많이 확보가능.

이 테스트셋은 살아있는 문서 처럼 시간이 지나며 계속 예시를 추가해야 합니다.

주의할 점 : 이 테스트셋에 맞춰 최적화하면 시스템이 이 테스트셋에 과의존하게 되어, 테스트는 다 잘 통과해도 새로운 유형의 질문에는 일반화가 안 될 수 있음. 그래서 최상의 테스트 데이터셋을 만드는데 시간을 투자하고, 문제가 발견될 때마다 계속 케이스를 추가하는 게 중요합니다.

## 2단계 : 검색(Retrieval) 성능 측정
관련 컨텐츠를 얼마나 잘 찾아오는가? 를 측정하는 지표들 (대표적인것만)

* MRR(Mean Reciprocal Rank, 평균 역순위) : 정답이 포함된 청크가 검색 결과에서 몇 번째로 나왔는지를 봄. 1위면 1점, 2위면 1/2점, 3위면 1/3점, 4위면 1/4점... 이걸 전체 질문에 대해 평균낸 것. MRR=1 이면 항상 첫 번째 청크에서 정답을 찾았다는 뜻(이상적)

* NDCG (Normalized Descounted Cumulative Gain, 정규화 할인 누적 이익) : 첫 번째 히트만 보는게 아니라 모든 관련 청크가 결과 전체에 얼마나 잘 분산/배치 되어 있는지를 봄. 로그가 들어간 공식이라 다소 복잡하지만, 핵심은 "관련 청크들이 상위권에 몰렸있으면 좋은점수".

* Recall @ k (재현율): 예를 들어 "Recall @ 3"이면, 상위 3개 청크만 봤을 때 테스트 케이스 중 몇 %가 정답 컨텍스트를 포함했는지. 50%면 테스트 케이스 절반은 상위 3개 안에 정답이 있었다는 뜻.
키워드가 여러 개일 때는 키워드 커버리지 비율(keyword coverage) 같은 변형 지표를 씀 — 키워드들이 여러 청크에 흩어질 수 있어서 조금 더 복잡함.

* Precision (정밀도): 노출된 청크 중 실제로 관련 있는 청크의 비율. K=5로 항상 5개를 보여준다면, 평균적으로 그 5개 중 몇 개가 실제로 관련 있었는지. "50% 정밀도"면 매번 5개 중 2.5개꼴로 관련 있는 청크였다는 뜻.
RAG에서는 Recall이 Precision보다 훨씬 중요합니다. 관련 없는 청크가 좀 섞여 들어가는 건 LLM이 어느정도 무시할 수 있지만, 정답이 애초에 검색 결과에 없으면 답을 낼 수 없기 때문. Precision은 "모델이 산만해지는 것(불필요한 정보로 시간 낭비)을 막고 싶을 때"에나 신경 쓰는 지표.

왜 검색 지표가 좋은가? : RAG가 실제로 뭘 하고 있는지 매우 직접적으로, 모델과 밀접하게 측정할 수 있음. 청크 크기 하나만 바꿔도 이 지표가 바로 반응하므로 빠르게 반복(Iterate) 가능

## 3단계 : 답변 (Answer) 자체의 품질 측정
참조 답변(reference answer)과 RAG 시스템이 실제로 생성한 답변을 비교합니다.

* 단순 단어 겸침 비교도 가능하지만 다소 부자연 스러움
* 벡터화해서 유사도 비교도 가능
* 가장 흔히 쓰는 방법 : "LLM을 판사로 쓰기(LLM-as-judge)" - 다른 (가능하면 더 강력한) LLM에게 "여기 참조 답변이 있고, 여기 시스템이 생성한 답변이 있다. 참조 답변과 비교해서 점수를 매겨라" 라고 요청. LLM은 이 작업을 꽤 잘 수행하며, 비교적 쉬운 테스크입니다.

평가 기준(1~5점 척도 등)으로 보통 세 가지를 봄:

정확성(Accuracy) - 맞는 답인가
완전성(Completeness) - 예: "Maxine"만으론 부족, "Maxine Thompson" 풀네임이 나와야 완전함
관련성(Relevance) - 완전성의 반대 개념. 질문과 무관한 정보(예: Maxine에 대한 다른 잡다한 정보)가 너무 많이 섞이면 관련성이 떨어짐

세 지표 모두 높아야 좋은 답변: 정확하고, 완전하고, 불필요한 정보 없이 관련성 있게.

## 검색 지표 vs 답변 지표 - 트레이드오프
검색 지표: 우리가 직접 통제 가능한 부분. 문제가 생기면 원인(어떤 컨텍스트가 노출됐는지 등)을 파악하기 쉬움. 모델과 가장 밀접하지만, 궁극적인 비즈니스 목표는 아님.
답변 지표: 최종 목표(좋은 답변)에 더 부합하지만, 문제 원인을 추적하기가 더 어려움.
그 아래엔 실제 사용자 피드백(예: ChatGPT의 "두 답변 중 어느 게 더 나은가요?" 같은 것) - 향후 개선을 위한 학습 데이터로 쓰이지만, 모델 정확도와 직접 연결짓기가 가장 어려움.

일반적인 워크플로: 검색 지표부터 측정 → 빠르게 반복해서 개선 → 그 다음 답변 품질 지표로 최종 목표 부합 여부 확인.


In [1]:
from evaluation import test

In [2]:
# JSON을 객체로 변환
tests = test.load_tests()

In [3]:
len(tests)

150

In [4]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)

2023년 권위 있는 IIOTY 상을 받은 사람은 누구인가요?
direct_fact
Maxine Thompson이 2023년 권위 있는 Insurellm 올해의 혁신가(IIOTY) 상을 수상했습니다.
['Maxine', 'Thompson', 'IIOTY']


In [5]:
from collections import Counter

count = Counter[str](t.category for t in tests)
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [6]:
from evaluation.eval import evaluate_answer, evaluate_retrieval

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.09523809523809523, ndcg=0.2222222222222222, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [8]:
eval, answer, chunks = evaluate_answer(example)

In [9]:
eval

AnswerEval(feedback="참조 답변에서는 수상자의 전체 이름인 'Maxine Thompson'을 명시했으나, 생성된 답변은 이름의 일부인 'Maxine'만 언급하여 정보가 불완전합니다. 상의 명칭을 정확히 언급한 점은 긍정적입니다.", accuracy=4.0, completeness=3.0, relevance=5.0)

In [10]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

참조 답변에서는 수상자의 전체 이름인 'Maxine Thompson'을 명시했으나, 생성된 답변은 이름의 일부인 'Maxine'만 언급하여 정보가 불완전합니다. 상의 명칭을 정확히 언급한 점은 긍정적입니다.
4.0
3.0
5.0
